# Markov Decision Processes (MDPs)

_Artificial Intelligence (CS221)_

**Planning when actions are unreliable. Sometimes you slip, so plan with the odds.**

In a search problem, an action always lands you where you expect. The real world is not so tidy.

     A Markov Decision Process handles randomness: an action might lead to different states by chance.

---

This notebook builds up the pieces of an MDP one step at a time — the transition model, the rewards, and the expected payoff of each action. Run each cell, read the short note above it, and let the idea settle before moving on. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python + NumPy

An MDP is described by a **transition model** (where actions take you, with what probability) and a **reward** for landing in each state. We build it in three steps: (1) the slippery transition tensor, (2) a check that it is a valid probability model, (3) the expected immediate reward of each action.

### Step 1 — Build the slippery transition model

There are three states `0, 1, 2` and two actions: `0 = left` and `1 = right`. The tensor `T[a, s, s']` holds the probability that taking action `a` in state `s` lands you in state `s'`. The world is **slippery**: even when the robot tries to go right, it sometimes stays put or drifts the other way, so each row is a spread of probabilities rather than a single certain outcome.

In [ ]:
# States 0, 1, 2.  Actions: 0 = "left", 1 = "right".
# T[a, s, s'] = probability of moving s -> s' under action a (the "slippery" robot).
T = np.array([
    [[0.8, 0.2, 0.0], [0.7, 0.3, 0.0], [0.0, 0.6, 0.4]],   # action left
    [[0.1, 0.9, 0.0], [0.0, 0.2, 0.8], [0.0, 0.0, 1.0]],   # action right
])

# Reward for arriving in each state — only the last state pays off.
R = np.array([0.0, 0.0, 5.0])

# Discount factor: future reward is worth a little less than reward now.
gamma = 0.9

### Step 2 — Confirm it is a valid probability model

For every action `a` and start state `s`, the probabilities over the next state `s'` must sum to exactly `1` — you always end up *somewhere*. Summing `T` along its last axis (`axis=2`) collapses the `s'` dimension and should leave an all-ones grid. This is a cheap sanity check that catches typos in the transition tensor.

In [ ]:
# Sum over the next-state axis (s'); every (action, start-state) entry must be 1.
row_sums = T.sum(axis=2)

print("row sums (must all be 1):")
print(row_sums)

### Step 3 — Expected immediate reward of each action

The expected reward of taking action `a` in state `s` is the average reward over where you might land: `sum over s' of T[a, s, s'] * R[s']`. Because `R` is a vector, the matrix product `T @ R` computes this for every `(action, state)` pair at once, giving a `[action, state]` table. This is the one-step payoff — full MDP planning then chains these together, discounted by `gamma`.

In [ ]:
# Expected immediate reward of action a in state s = sum_s' T[a,s,s'] * R[s'].
# T @ R contracts the last axis, leaving one value per [action, state].
exp_reward = T @ R

for a, name in enumerate(["left ", "right"]):
    print("action", name, "-> expected reward by start state:",
          np.round(exp_reward[a], 2))

print("discount gamma =", gamma)

## Visualize the data & results

_For a warehouse picking robot advancing down an aisle, what immediate reward does each move earn on average?_

Same idea as above, but with a concrete story: the states are positions along an aisle and the actions are `advance` / `retreat`. We compute the expected immediate reward for each move and draw it as a heatmap.

### Step 1 — Set up the warehouse MDP and its expected rewards

The three states are `aisle start`, `mid-aisle`, and `shelf`; the two actions are `advance` and `retreat`. Only reaching the shelf pays a reward of `1`. As before, `T @ R` gives the expected immediate reward for every `[action, state]` pair.

In [ ]:
# States: aisle-start, mid-aisle, shelf.  Actions: advance, retreat.
T = np.array([
    [[0.8, 0.2, 0.0], [0.0, 0.7, 0.3], [0.0, 0.0, 1.0]],   # advance
    [[1.0, 0.0, 0.0], [0.6, 0.4, 0.0], [0.0, 0.6, 0.4]],   # retreat
])

# Reward for reaching the shelf.
R = np.array([0.0, 0.0, 1.0])

# Expected immediate reward, indexed [action, state].
exp_reward = T @ R

### Step 2 — Draw the expected-reward heatmap

Each cell shows the average immediate reward of a move from a given spot. Brighter means more reward. Advancing from mid-aisle is worth the most because it has a real chance of reaching the rewarding shelf in a single step.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))

im = ax.imshow(exp_reward, cmap="viridis", aspect="auto")

# Label the action rows and state columns.
ax.set_yticks([0, 1])
ax.set_yticklabels(["advance", "retreat"])
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(["aisle start", "mid-aisle", "shelf"])

# Annotate each cell with its numeric reward.
for a in range(2):
    for s in range(3):
        ax.text(s, a, round(exp_reward[a, s], 1), ha="center", va="center", color="white")

ax.set_title("warehouse robot: expected immediate reward")
fig.colorbar(im, ax=ax)
plt.show()